# 14 · Consolidación del grupo **DEMOGRAPHICS**

**Proyecto:** Modelo de longevidad NHANES 2017-2018 (ciclo `_J`)
**Capa:** `01_raw` (.xpt) → **`02_intermediate`** → `03_primary`

## Objetivo
Consolidar `DEMO_J` en un dataset intermedio compacto de **covariables** (demografía y nivel socioeconómico). DEMO **no tiene target propio**: aporta edad/sexo (necesarios para calcular PhenoAge al unir con `laboratory`) y actúa como spine demográfico del resto de datasets.

## Decisiones clave
1. **Recodificar códigos especiales NHANES** (`7/77` = rechazó, `9/99` = no sabe) a `NaN` antes de cualquier cálculo.
2. **Descartar variables de diseño muestral** (`WTINT2YR`, `WTMEC2YR`, `SDMVPSU`, `SDMVSTRA`), composición del hogar, idioma/proxy de entrevista y duplicados (`RIDRETH1` vs `RIDRETH3`): ruido para el modelado predictivo.
3. **Seleccionar el núcleo socioeconomico-demográfico:** `edad`, `sexo`, `raza`, `educacion` (ordinal), `estado_civil`, `nacido_eeuu`, `pir` (ratio ingreso/pobreza).
4. **Edad top-coded en 80** (límite NHANES): se conserva como está (trampa conocida).

> NO se imputa ni se escala aquí (eso es post-split, en los pipelines `data_imputation`/`data_encoding`).

## 1. Setup

In [1]:
%load_ext kedro.ipython

[06/20/26 14:13:42] INFO     Using                                                                  ]8;id=9797390;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=9797391;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\framework\project\__init__.py#281\281]8;;\
                             'C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages                
                             \kedro\framework\project\rich_logging.yml' as logging configuration.                  

                    INFO     Registered line magic '%reload_kedro'                                   ]8;id=9797398;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=9797399;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py#67\67]8;;\

                    INFO     Registered line magic '%load_node'                                      ]8;id=9797405;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=9797406;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py#69\69]8;;\

                    INFO     Resolved project path as:                                              ]8;id=9797412;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=9797413;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py#193\193]8;;\
                             C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes.                                        
                             To set a different path, run '%reload_kedro <project_root>'                           

[06/20/26 14:13:44] INFO     No typed parameter requirements found, returning original   ]8;id=9797420;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\validation\parameter_validator.py\parameter_validator.py]8;;\:]8;id=9797421;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\validation\parameter_validator.py#124\124]8;;\
                             parameters                                                                            

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=9797428;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=9797429;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro_telemetry\plugin.py#273\273]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

[06/20/26 14:13:45] INFO     Kedro project Nhanes                                                   ]8;id=9797435;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=9797436;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py#159\159]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=9797442;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=9797443;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\ipython\__init__.py#160\160]8;;\
                             'pipelines'                                                                           

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 120)

PROJ = Path(context.project_path)
OUT_DIR = PROJ / "data" / "02_intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Salida:", OUT_DIR)

Salida: C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\data\02_intermediate


## 2. Carga de DEMO_J

In [3]:
demo = catalog.load("demo_j_dataset")
demo = demo.drop_duplicates("SEQN").copy()
demo["SEQN"] = demo["SEQN"].astype("int64")
demo = demo.set_index("SEQN")
print("DEMO_J:", demo.shape)

                    INFO     Loading data from demo_j_dataset (GenericDataset)...              ]8;id=9797450;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=9797451;file://C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\.venv\Lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

DEMO_J: (9254, 45)


## 3. Recodificación de códigos especiales → `NaN`

NHANES codifica no-respuestas como números altos. Si no se limpian, contaminan el escalado y la imputación posteriores.

In [4]:
def s_nan(series, bad):
    # Convierte a NaN los codigos especiales (rechazo / no sabe)
    return series.where(~series.isin(bad))

BAD_YN = [7, 9]      # items de 1 digito (rechazo=7, no sabe=9)
BAD_2  = [77, 99]    # items de 2 digitos
print("helpers listos")

helpers listos


## 4. Selección y derivación de covariables

Cada variable cruda se mapea a una columna interpretable. Las categóricas se dejan como texto (el encoding va después); las ordinales/continuas como número.

In [5]:
F = {}  # diccionario de columnas finales (Series indexadas por SEQN)

# --- Edad (continua, top-coded en 80). round() corrige el artefacto de
#     read_sas/xport que lee el 0 (infantes <1 año) como ~5e-79; son años enteros ---
F["edad"] = demo["RIDAGEYR"].astype(float).round(0)

# --- Sexo (binaria) ---
F["sexo"] = demo["RIAGENDR"].map({1: "masculino", 2: "femenino"}).astype("object")

# --- Raza/etnia (nominal, RIDRETH3 es mas granular que RIDRETH1) ---
raza_map = {
    1: "mexicano_americano", 2: "otro_hispano", 3: "blanco_no_hispano",
    4: "negro_no_hispano", 6: "asiatico_no_hispano", 7: "otro_multirracial",
}
F["raza"] = demo["RIDRETH3"].map(raza_map).astype("object")

# --- Educacion del adulto (ORDINAL 1<...<5, se deja numerica para KNN) ---
# 1=<9no  2=9-11  3=bachillerato/GED  4=universidad parcial/AA  5=universitario+
F["educacion"] = s_nan(demo["DMDEDUC2"], BAD_YN).astype(float)

# --- Estado civil (nominal) ---
civil_map = {
    1: "casado", 2: "viudo", 3: "divorciado", 4: "separado",
    5: "nunca_casado", 6: "union_libre",
}
F["estado_civil"] = s_nan(demo["DMDMARTL"], BAD_2).map(civil_map).astype("object")

# --- Nacido en EEUU (binaria, DMDBORN4: 1=EEUU 2=otro) ---
F["nacido_eeuu"] = s_nan(demo["DMDBORN4"], BAD_2).map({1: "si", 2: "no"}).astype("object")

# --- Ratio ingreso familiar / umbral de pobreza (continua, 0-5; 5 = >=5) ---
F["pir"] = demo["INDFMPIR"].astype(float)

print({k: str(v.dtype) for k, v in F.items()})

{'edad': 'float64', 'sexo': 'object', 'raza': 'object', 'educacion': 'float64', 'estado_civil': 'object', 'nacido_eeuu': 'object', 'pir': 'float64'}


## 5. Ensamblado y control de calidad

In [6]:
demo_int = pd.DataFrame(F)
demo_int = demo_int[["edad", "sexo", "raza", "educacion",
                     "estado_civil", "nacido_eeuu", "pir"]]
print("shape:", demo_int.shape)
print("\n--- % nulos por columna ---")
print((demo_int.isnull().mean() * 100).round(1))
print("\n--- dtypes ---")
print(demo_int.dtypes)

shape: (9254, 7)

--- % nulos por columna ---
edad             0.0
sexo             0.0
raza             0.0
educacion       40.0
estado_civil    39.9
nacido_eeuu      0.0
pir             13.3
dtype: float64

--- dtypes ---
edad            float64
sexo             object
raza             object
educacion       float64
estado_civil     object
nacido_eeuu      object
pir             float64
dtype: object


In [7]:
print("edad:\n", demo_int["edad"].describe())
for c in ["sexo", "raza", "estado_civil", "nacido_eeuu"]:
    print(f"\n{c}:")
    print(demo_int[c].value_counts(dropna=False))
print("\neducacion:\n", demo_int["educacion"].value_counts(dropna=False).sort_index())

edad:
 count    9254.000000
mean       34.334234
std        25.500280
min         0.000000
25%        11.000000
50%        31.000000
75%        58.000000
max        80.000000
Name: edad, dtype: float64

sexo:
sexo
femenino     4697
masculino    4557
Name: count, dtype: int64

raza:
raza
blanco_no_hispano      3150
negro_no_hispano       2115
mexicano_americano     1367
asiatico_no_hispano    1168
otro_hispano            820
otro_multirracial       634
Name: count, dtype: int64

estado_civil:
estado_civil
NaN             3691
casado          2737
nunca_casado    1006
divorciado       641
union_libre      515
viudo            462
separado         202
Name: count, dtype: int64

nacido_eeuu:
nacido_eeuu
si     7303
no     1948
NaN       3
Name: count, dtype: int64

educacion:
 educacion
1.0     479
2.0     638
3.0    1325
4.0    1778
5.0    1336
NaN    3698
Name: count, dtype: int64


## 6. Export a `02_intermediate`

SEQN viaja en el índice (llave para unir DEMO como spine). NO se imputa ni se escala.

In [8]:
demo_int.index.name = "SEQN"
out_path = OUT_DIR / "demo_intermediate.parquet"
demo_int.to_parquet(out_path)
print("Guardado:", out_path)
print("Verificacion -> shape:", pd.read_parquet(out_path).shape)

Guardado: C:\Users\Felipe\Desktop\NHANES_PROJECT\nhanes\data\02_intermediate\demo_intermediate.parquet
Verificacion -> shape: (9254, 7)
